### Transformers – Lösung Schritt für Schritt

*44. Bundeswettbewerb Informatik (2025/2026), 2. Runde, Aufgabe 1*

Fred hat zum Geburtstag einen großen Eimer kreisförmiger Steckplättchen geschenkt bekommen. Man kann sie durch kleine Stifte
miteinander verbinden. An jedem Plättchen gibt es vier Kontaktpunkte, jeweils nach 90°, an die insgesamt bis zu vier Stifte gesteckt
werden können. Die Steckverbindungen erlauben es, die Plättchen um 180° um eine Stiftachse zu drehen. Aus Plättchen und Stiften lassen
sich viele verschiedene Figuren bauen. Durch Drehungen an den Stiftachsen kann Fred eine Figur in andere Figuren überführen.

Fred fragt sich, ob er für zwei verschiedene Figuren entscheiden kann, ob sie sich durch derartige Drehungen ineinander überführen
lassen. Das Bild zeigt ein Beispiel: Die Figur links kann in die Figur oben rechts, aber nicht in die Figur unten rechts überführt
werden.

#### Aufgabe

Hilf Fred und schreibe ein Programm, das für zwei Figuren entscheidet, ob und, wenn ja, wie die eine in die andere Figur überführt
werden kann.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `transform0y.txt`, `transform0n.txt` und `transform01.txt` bis `transform13.txt`. Jede Datei enthält
zuerst die Ausgangsfigur, dann die Zielfigur. Jede Figur steht als Raster aus Zeilen und Spalten:

- 1. Zeile: die Zahlen `s` und `z`, die Zahl der Spalten und der Zeilen, die Plättchen enthalten.
- Dann die Darstellung: `o` ist ein Plättchen, `.` ein leerer Platz, `|` ein senkrechter und `-` ein waagerechter Stift.
- Eine Leerzeile.

Das Negativbeispiel vom Aufgabenblatt (`transform0n.txt`):

```
3 2
..o-o
..|..
o-o-o

3 3
..o..
..|..
..o..
..|..
o-o-o
```

#### Ausgabeformat

1. Zeile: `y` (möglich) oder `n` (nicht möglich).
2. Zeile: die Zahl der benötigten Drehungen (diese und die folgenden Zeilen nur bei „möglich“).
3. Dann für jede Drehung die Figur **vor** der Drehung im Format der Eingabe, wobei das Plättchen, **um das gedreht wird**, mit `#`
   statt `o` markiert ist, und eine Leerzeile.

Die 1. und 2. Zeile der Ausgabe müssen für alle Beispiele in der Dokumentation stehen, die vollständige Ausgabe für `transform0n`,
`transform0y`, `transform01`, `transform02`, `transform03`, `transform05` und `transform10`.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graph:** Plättchen sind **Knoten**, Stifte sind **Kanten**. Die Figuren liegen auf einem Gitter, jede Kante verbindet zwei
  benachbarte Gitterpunkte.
- **Brücke:** Eine Kante, bei deren Entfernen die Figur in zwei Teile zerfällt. Nur an einer Brücke kann man eine Teilfigur drehen,
  denn bei einem Stift, der in einem Kreis liegt, hängt die Figur auch auf der anderen Seite fest.
- **Zusammenhangskomponente:** Alle Knoten, die man von einem Knoten aus erreicht, wenn man den Kanten folgt. Man findet sie mit einer
  Suche, die jeden Knoten nur einmal besucht (*Flood Fill*).
- **Spiegeln:** Eine Drehung um 180° um einen Stift ist für Figuren in der Ebene eine **Spiegelung an der Geraden**, auf der der Stift
  liegt. Bei einem waagerechten Stift auf der Höhe `y0` wird aus `(x, y)` der Punkt `(x, 2 · y0 − y)`.
- **Zustandsraum und Breitensuche:** Jede Figur ist ein Zustand, jede Drehung ein Übergang. Die Breitensuche besucht alle
  Zustände nach der Zahl der Drehungen geordnet. Besuchte Zustände merkt man sich in einer Menge, damit man sie nicht doppelt
  untersucht. Dafür müssen Zustände in Python **unveränderlich** (`frozenset`, Tupel) sein.
- **Invarianten:** Eigenschaften, die sich bei einer Drehung nie ändern, zum Beispiel die Zahl der Plättchen und Stifte oder die
  Verteilung der Nachbarn. Sind sie bei zwei Figuren verschieden, geht es **sicher nicht**. Das spart die Suche.
- **Farbverfeinerung:** Man färbt jedes Plättchen nach seiner Umgebung (wie viele waagerechte und senkrechte Nachbarn) und verfeinert
  die Farben immer weiter, indem man die Farben der Nachbarn mit einbezieht. Haben zwei Figuren nicht dieselbe Verteilung der Farben,
  sind sie nicht überführbar.
- **Normalform durch „Aufklappen“:** Wenn man beide Figuren auf dieselbe einfache Form bringt, hat man einen Weg: erst von der
  Ausgangsfigur zur Normalform, dann rückwärts von der Normalform zur Zielfigur. Alle Drehungen lassen sich umkehren.

**Nützliche Funktionen und Bibliotheken**
- `set` und `frozenset`, `dict`, Tupel als Schlüssel, `sorted`
- `collections.deque` als Warteschlange (`append`, `popleft`)

## Die Idee

1. Die Figuren einlesen und als **Menge von Plättchen und Stiften** speichern. Die Lage im Raster spielt keine Rolle, nur die Form.
   Deshalb schieben wir jede Figur in die obere linke Ecke (*normieren*), dann sind gleiche Figuren auch gleiche Daten.
2. Alle **Drehungen** einer Figur erzeugen: an jeder Brücke jede der beiden Seiten spiegeln, wenn sich dabei keine Plättchen überlappen.
3. **Schnell ausschließen**, was sicher nicht geht (Zählen, Farbverfeinerung).
4. Bei **kleinen** Figuren die Breitensuche: Sie findet einen Weg oder beweist, dass es keinen gibt.
5. Bei **größeren** Figuren beide Figuren „aufklappen“, bis eine Normalform erreicht ist. Stimmen die Normalformen überein, setzt man
   den Weg zusammen. Sonst bleibt die Frage offen.
6. Den Weg im vorgeschriebenen Format ausgeben und selbst **nachprüfen**.

**Welche Drehung?** Die Aufgabe lässt zwei Auslegungen zu. Wir drehen eine Teilfigur um den **Stift**, an dem sie hängt (*Kantendrehung*).
Das Plättchen, das am Stift auf der gedrehten Seite sitzt, ist das markierte. So zeigt es auch die Beispielausgabe `transform02_out.txt`.
Die andere Auslegung (*Knotendrehung*, das Plättchen wird zwischen zwei gegenüberliegenden Stiften gedreht) steht unter „Für Neugierige“.

## Schritt 1: Figuren einlesen und zeichnen

Eine **Figur** speichern wir als Paar `(plaettchen, stifte)`:

- `plaettchen` ist eine Menge von Rasterpunkten `(x, y)`. Das Plättchen in der `x`-ten Plättchen-Spalte und `y`-ten Plättchen-Zeile steht
  im Text an Position `(2 · x, 2 · y)`, denn zwischen zwei Plättchen ist Platz für einen Stift.
- `stifte` ist eine Menge von Paaren zweier Plättchen. Ein `-` zwischen `(x, y)` und `(x + 1, y)` ist der Stift `((x, y), (x + 1, y))`,
  ein `|` zwischen `(x, y)` und `(x, y + 1)` der Stift `((x, y), (x, y + 1))`.

Beides sind `frozenset`s, also Mengen, die man nicht mehr verändern kann. Nur dann kann man eine ganze Figur als Schlüssel in einem
`dict` oder als Element einer Menge benutzen. `normiere` verschiebt die Figur so, dass das kleinste `x` und das kleinste `y` null
sind. Zwei Figuren, die sich nur durch Verschieben unterscheiden, sind dann **gleich**.

`zeichne` macht aus einer Figur wieder den Text im Format der Eingabe, optional mit einem markierten Plättchen (`#`).

In [ ]:
def normiere(plaettchen, stifte):
    kleinstes_x = min(x for x, y in plaettchen)
    kleinstes_y = min(y for x, y in plaettchen)
    neue_plaettchen = set()
    for x, y in plaettchen:
        neue_plaettchen.add((x - kleinstes_x, y - kleinstes_y))
    neue_stifte = set()
    for a, b in stifte:
        a = (a[0] - kleinstes_x, a[1] - kleinstes_y)
        b = (b[0] - kleinstes_x, b[1] - kleinstes_y)
        neue_stifte.add((min(a, b), max(a, b)))
    return frozenset(neue_plaettchen), frozenset(neue_stifte)


def figur_aus_text(zeilen):
    plaettchen = set()
    stifte = set()
    for y in range(len(zeilen)):
        for x in range(len(zeilen[y])):
            zeichen = zeilen[y][x]
            if zeichen == 'o' or zeichen == '#':
                plaettchen.add((x // 2, y // 2))
            elif zeichen == '-':
                stifte.add(((x // 2, y // 2), (x // 2 + 1, y // 2)))
            elif zeichen == '|':
                stifte.add(((x // 2, y // 2), (x // 2, y // 2 + 1)))
    return normiere(plaettchen, stifte)


def lies_figuren(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().split('\n')
    figuren = []
    i = 0
    while i < len(zeilen):
        zeilen[i] = zeilen[i].rstrip()
        if zeilen[i] == '':
            i = i + 1
            continue
        spalten, reihen = zeilen[i].split()
        anzahl_zeilen = 2 * int(reihen) - 1                # Plättchen-Zeilen und die Zeilen mit den senkrechten Stiften dazwischen
        figuren.append(figur_aus_text(zeilen[i + 1:i + 1 + anzahl_zeilen]))
        i = i + 1 + anzahl_zeilen
    return figuren[0], figuren[1]


def zeichne(figur, markiert=None):
    plaettchen, stifte = figur
    breite = max(x for x, y in plaettchen) + 1
    hoehe = max(y for x, y in plaettchen) + 1
    raster = []
    for zeile in range(2 * hoehe - 1):
        raster.append(['.'] * (2 * breite - 1))
    for x, y in plaettchen:
        if (x, y) == markiert:
            raster[2 * y][2 * x] = '#'
        else:
            raster[2 * y][2 * x] = 'o'
    for a, b in stifte:
        if a[1] == b[1]:
            raster[2 * a[1]][2 * a[0] + 1] = '-'
        else:
            raster[2 * a[1] + 1][2 * a[0]] = '|'
    zeilen = [str(breite) + ' ' + str(hoehe)]
    for zeile in raster:
        zeilen.append(''.join(zeile))
    return '\n'.join(zeilen)


start, ziel = lies_figuren('beispieldaten/transform0n.txt')
print(zeichne(start))
print()
print(zeichne(ziel))
print()
print('Plättchen der Ausgangsfigur:', sorted(start[0]))
print('Stifte der Ausgangsfigur:', sorted(start[1]))

3 2
..o-o
..|..
o-o-o

3 3
..o..
..|..
..o..
..|..
o-o-o

Plättchen der Ausgangsfigur: [(0, 1), (1, 0), (1, 1), (2, 0), (2, 1)]
Stifte der Ausgangsfigur: [((0, 1), (1, 1)), ((1, 0), (1, 1)), ((1, 0), (2, 0)), ((1, 1), (2, 1))]


`zeichne` gibt dieselbe Darstellung wie die Eingabe zurück (und `normiere` hat die Figur nach links oben geschoben). Die Ausgangsfigur
von `transform0n.txt` hat 5 Plättchen und 4 Stifte.

## Schritt 2: Der Graph und seine Brücken

Für die Suche nach Nachbarn bauen wir ein `dict` `nachbarn`: Zu jedem Plättchen die Liste der Plättchen, mit denen es durch einen Stift
verbunden ist.

`seite(nachbarn, u, v)` findet alle Plättchen, die man von `v` aus erreicht, **ohne den Stift `(u, v)` zu benutzen**. Das ist ein
Flood Fill: Wir haben eine Liste `offen` mit noch nicht untersuchten Plättchen und eine Menge `gesehen` mit schon bekannten. Wir
nehmen ein Plättchen aus `offen`, und alle Nachbarn, die noch nicht bekannt sind (und nicht über den verbotenen Stift erreicht werden),
kommen dazu.

Damit erkennen wir auch **Brücken**: Erreicht man `u` von `v` aus nicht, ohne den Stift zu benutzen, ist `(u, v)` eine Brücke. Und
`seite` ist dann genau die Teilfigur auf der Seite von `v`.

In [ ]:
def baue_nachbarn(figur):
    plaettchen, stifte = figur
    nachbarn = {}
    for p in plaettchen:
        nachbarn[p] = []
    for a, b in stifte:
        nachbarn[a].append(b)
        nachbarn[b].append(a)
    return nachbarn


def seite(nachbarn, u, v):
    """Alle Plättchen, die man von v aus erreicht, ohne den Stift zwischen u und v zu benutzen."""
    gesehen = {v}
    offen = [v]
    while len(offen) > 0:
        k = offen.pop()
        for w in nachbarn[k]:
            if k == v and w == u:
                continue                                   # der verbotene Stift
            if w not in gesehen:
                gesehen.add(w)
                offen.append(w)
    return gesehen


nachbarn = baue_nachbarn(start)
for a, b in sorted(start[1]):
    ist_bruecke = a not in seite(nachbarn, a, b)
    print('Stift', a, b, ': Brücke?', ist_bruecke, ', Teil hinter dem Stift:', sorted(seite(nachbarn, a, b)))

Stift (0, 1) (1, 1) : Brücke? True , Teil hinter dem Stift: [(1, 0), (1, 1), (2, 0), (2, 1)]
Stift (1, 0) (1, 1) : Brücke? True , Teil hinter dem Stift: [(0, 1), (1, 1), (2, 1)]
Stift (1, 0) (2, 0) : Brücke? True , Teil hinter dem Stift: [(2, 0)]
Stift (1, 1) (2, 1) : Brücke? True , Teil hinter dem Stift: [(2, 1)]


In dieser Figur ist jeder Stift eine Brücke, denn sie hat keinen Kreis. Hinter jedem Stift, von `a` aus gesehen, liegt ein anderer Teil
der Figur.

## Schritt 3: Alle Drehungen einer Figur

Zu jeder Brücke `(a, b)` gibt es zwei Möglichkeiten: Die Seite von `b` wird gedreht (und `a` bleibt), oder umgekehrt. Wir nennen den
festen Plättchen `u` und das auf der gedrehten Seite `v` (`v` ist das **markierte** Plättchen).

Die gedrehte Seite wird an der Geraden durch den Stift gespiegelt:

- Liegt der Stift **waagerecht** (beide Plättchen haben dasselbe `y = y0`), wird aus `(x, y)` der Punkt `(x, 2 · y0 − y)`.
- Liegt er **senkrecht** (dasselbe `x = x0`), wird aus `(x, y)` der Punkt `(2 · x0 − x, y)`.

`v` selbst liegt auf der Geraden und bleibt, wo es ist. Die Drehung ist nur erlaubt, wenn **kein gespiegeltes Plättchen auf einem Plättchen
des festen Teils landet**. Von den Stiften werden die Enden gespiegelt, die auf der gedrehten Seite liegen. Danach wird normiert.

`alle_zuege(figur)` gibt eine Liste von Paaren `(neue_figur, markiertes_plaettchen)` zurück, wobei das markierte Plättchen in den
Koordinaten der alten Figur angegeben ist.

In [ ]:
def spiegele(p, u, v):
    """Spiegelt den Punkt p an der Geraden durch u und v."""
    if u[1] == v[1]:                                       # waagerechter Stift: an der Höhe u[1] spiegeln
        return (p[0], 2 * u[1] - p[1])
    return (2 * u[0] - p[0], p[1])                         # senkrechter Stift: an der Spalte u[0] spiegeln


def alle_zuege(figur):
    plaettchen, stifte = figur
    nachbarn = baue_nachbarn(figur)
    ergebnis = []
    for a, b in sorted(stifte):
        for u, v in [(a, b), (b, a)]:                      # die Seite von v wird gedreht, u bleibt
            gedreht = seite(nachbarn, u, v)
            if u in gedreht:
                break                                      # kein Brücke: ein Kreis hält die Figur zusammen
            fest = plaettchen - gedreht
            neue_plaettchen = set()
            for p in gedreht:
                neue_plaettchen.add(spiegele(p, u, v))
            if len(neue_plaettchen & fest) > 0:
                continue                                   # Überlappung: die Drehung ist nicht möglich
            neue_stifte = set()
            for s, t in stifte:
                if s in gedreht:
                    s = spiegele(s, u, v)
                if t in gedreht:
                    t = spiegele(t, u, v)
                neue_stifte.add((min(s, t), max(s, t)))
            ergebnis.append((normiere(fest | neue_plaettchen, neue_stifte), v))
    return ergebnis


zuege = alle_zuege(start)
print('Mögliche Drehungen der Ausgangsfigur von transform0n.txt:', len(zuege))
for neue_figur, markiert in zuege:
    print('Markiertes Plättchen', markiert)
    print(zeichne(neue_figur, None))
    print()

Mögliche Drehungen der Ausgangsfigur von transform0n.txt: 8
Markiertes Plättchen (1, 1)
3 2
o-o-o
..|..
..o-o

Markiertes Plättchen (0, 1)
3 2
..o-o
..|..
o-o-o

Markiertes Plättchen (1, 1)
3 2
..o-o
..|..
o-o-o

Markiertes Plättchen (1, 0)
3 2
o-o..
..|..
o-o-o

Markiertes Plättchen (2, 0)
3 2
..o-o
..|..
o-o-o

Markiertes Plättchen (1, 0)
3 2
o-o-o
..|..
..o-o

Markiertes Plättchen (2, 1)
3 2
..o-o
..|..
o-o-o

Markiertes Plättchen (1, 1)
3 2
o-o-o
..|..
..o-o



Die Ausgangsfigur hat vier Stifte und an jedem lassen sich zwei Seiten drehen, das sind acht Drehungen, und hier sind alle erlaubt.
Einige ergeben dieselbe Figur wie die Ausgangsfigur: Dreht man ein einzelnes Plättchen am Ende eines Stifts, liegt es ja auf der
Achse und bleibt, wo es ist. Solche Drehungen schaden nicht, die Suche merkt, dass sie nichts Neues bringen. Bei Drehungen von
größeren Teilen ändert sich die Figur.

## Schritt 4: Breitensuche bei kleinen Figuren

Die Breitensuche beginnt bei der Ausgangsfigur und erzeugt alle Figuren, die man mit 1, 2, 3, … Drehungen erreicht. Eine
**Warteschlange** (`deque`) hält die Figuren, die noch untersucht werden müssen. In `vorgaenger` merken wir für jede schon gesehene
Figur, aus welcher sie entstanden ist. Das ist zugleich die Menge der besuchten Figuren und hilft später, den Weg zurückzuverfolgen.

Es gibt drei Ausgänge:

- Die Zielfigur wird erreicht: **ja**, und der Weg steht in `vorgaenger`.
- Die Warteschlange wird leer, ohne dass die Zielfigur vorkam: Dann haben wir **alle** erreichbaren Figuren gesehen, es geht
  sicher **nicht**.
- Es wurden mehr Figuren gesehen als erlaubt (`grenze`): **unbekannt**. Das schützt vor einer endlosen Suche.

Wir rechnen mit den Figuren selbst als Schlüsseln, deshalb ist es so wichtig, dass sie normiert sind.

In [ ]:
from collections import deque


def breitensuche(start, ziel, grenze):
    """Ergebnis: ('ja', Liste der Figuren von start bis ziel), ('nein', None) oder ('unbekannt', None)."""
    vorgaenger = {start: None}
    warteschlange = deque([start])
    while len(warteschlange) > 0:
        figur = warteschlange.popleft()
        for neue_figur, markiert in alle_zuege(figur):
            if neue_figur not in vorgaenger:
                vorgaenger[neue_figur] = figur
                if neue_figur == ziel:
                    weg = [ziel]
                    while vorgaenger[weg[-1]] is not None:
                        weg.append(vorgaenger[weg[-1]])
                    weg.reverse()
                    return 'ja', weg
                warteschlange.append(neue_figur)
        if len(vorgaenger) > grenze:
            return 'unbekannt', None
    return 'nein', None


for name in ['0y', '0n', '01', '02']:
    start, ziel = lies_figuren('beispieldaten/transform' + name + '.txt')
    urteil, weg = breitensuche(start, ziel, 100000)
    if urteil == 'ja':
        print('transform' + name + ':', urteil, ', Drehungen:', len(weg) - 1)
    else:
        print('transform' + name + ':', urteil)

transform0y: ja , Drehungen: 1
transform0n: nein
transform01: nein
transform02: ja , Drehungen: 3


Für `transform0y.txt` genügt eine Drehung, für `transform02.txt` sind es drei. Bei `transform0n.txt` und `transform01.txt` hat die
Suche alle erreichbaren Figuren durchgesehen, ohne die Zielfigur zu finden: **Es geht nicht.**

Wie viele Figuren gibt es? Das hängt stark von der Figur ab. Schon einfache Figuren haben exponentiell viele: Hat eine Figur `k` Blätter,
die man unabhängig voneinander umklappen kann, sind es `2^k` Figuren. Bei Figuren mit mehr als etwa 20 bis 30 Plättchen ist die
Breitensuche meist zu langsam.

## Schritt 5: Schnell ausschließen mit Invarianten

Bevor wir suchen, prüfen wir Eigenschaften, die sich bei einer Drehung nicht ändern. Sind sie bei beiden Figuren verschieden, geht es
nicht, und wir sparen die Suche. Das ist besonders bei großen Figuren wichtig, bei denen die Suche ohnehin nicht klappt.

1. **Zählen:** Die Zahl der Plättchen und der Stifte bleibt gleich.
2. **Farbverfeinerung:** Jedes Plättchen bekommt zuerst als „Farbe“ das Paar (Zahl der waagerechten Nachbarn, Zahl der senkrechten
   Nachbarn). Diese beiden Zahlen ändern sich bei einer Drehung nicht, denn ein waagerechter Stift bleibt waagerecht und ein
   senkrechter senkrecht. In jeder Runde bekommt jedes Plättchen eine **neue Farbe**, die aus der alten Farbe und den (sortierten)
   Farben seiner waagerechten und seiner senkrechten Nachbarn besteht. Die neuen Farben ersetzen wir durch kleine Zahlen. So
   unterscheiden sich Plättchen immer feiner nach ihrer weiteren Umgebung.

Eine Drehung ordnet die Plättchen nur um. Darum muss bei überführbaren Figuren **jede Farbe gleich oft** vorkommen. Wir färben beide
Figuren gemeinsam, damit gleiche Farben dasselbe bedeuten, und vergleichen nach jeder Runde die **sortierten Listen der Farben** der
beiden Figuren. Sind sie verschieden, hören wir sofort auf. Die Verfeinerung hört auf, wenn eine Runde keine neuen Farben mehr
ergibt.

Die Invarianten können nur **ausschließen**. Sind sie gleich, folgt daraus noch nicht, dass die Figuren überführbar sind.

In [ ]:
def sind_vielleicht_ueberfuehrbar(figur_a, figur_b):
    """Falsch: die Figuren sind sicher nicht überführbar. Wahr: die Invarianten sprechen nicht dagegen."""
    if len(figur_a[0]) != len(figur_b[0]) or len(figur_a[1]) != len(figur_b[1]):
        return False
    # gemeinsame Farbverfeinerung: ein Knoten ist (nummer der Figur, plättchen)
    waagerecht = {}
    senkrecht = {}
    for nr in [0, 1]:
        figur = [figur_a, figur_b][nr]
        for p in figur[0]:
            waagerecht[(nr, p)] = []
            senkrecht[(nr, p)] = []
        for a, b in figur[1]:
            if a[1] == b[1]:
                waagerecht[(nr, a)].append((nr, b))
                waagerecht[(nr, b)].append((nr, a))
            else:
                senkrecht[(nr, a)].append((nr, b))
                senkrecht[(nr, b)].append((nr, a))
    farbe = {}
    for k in waagerecht:
        farbe[k] = (len(waagerecht[k]), len(senkrecht[k]))
    anzahl_farben = 0
    while True:
        neu = {}
        for k in waagerecht:
            w = sorted([farbe[n] for n in waagerecht[k]])
            s = sorted([farbe[n] for n in senkrecht[k]])
            neu[k] = (farbe[k], tuple(w), tuple(s))
        namen = {}
        for wert in sorted(set(neu.values())):
            namen[wert] = len(namen)
        farbe = {}
        for k in neu:
            farbe[k] = namen[neu[k]]
        farben_a = sorted([farbe[k] for k in farbe if k[0] == 0])
        farben_b = sorted([farbe[k] for k in farbe if k[0] == 1])
        if farben_a != farben_b:
            return False                                   # die Farben sind verschieden verteilt
        if len(namen) == anzahl_farben:
            return True                                    # keine neuen Farben mehr
        anzahl_farben = len(namen)


for name in ['0y', '0n', '01', '02', '03', '04']:
    start, ziel = lies_figuren('beispieldaten/transform' + name + '.txt')
    print('transform' + name + ': Plättchen', len(start[0]), ', Invarianten sprechen nicht dagegen:', sind_vielleicht_ueberfuehrbar(start, ziel))

transform0y: Plättchen 5 , Invarianten sprechen nicht dagegen: True
transform0n: Plättchen 5 , Invarianten sprechen nicht dagegen: False
transform01: Plättchen 5 , Invarianten sprechen nicht dagegen: False
transform02: Plättchen 11 , Invarianten sprechen nicht dagegen: True
transform03: Plättchen 14 , Invarianten sprechen nicht dagegen: True
transform04: Plättchen 2006 , Invarianten sprechen nicht dagegen: False


Bei `transform0n.txt`, bei `transform01.txt` und bei `transform04.txt` mit über 2000 Plättchen **beweisen die Invarianten sofort**,
dass es nicht geht. Bei `transform03.txt` und den anderen sprechen sie nicht dagegen, da muss gesucht werden.

## Schritt 6: Größere Figuren, Aufklappen bis zur Normalform

Bei größeren Figuren ist die Breitensuche zu langsam. Wir nutzen dann, dass **jede Drehung umkehrbar** ist (dreht man dieselbe Teilfigur
noch einmal am selben Stift, ist man wieder zurück). Wenn wir beide Figuren auf **dieselbe** Figur bringen, die **Normalform**, dann
gibt es einen Weg: von der Ausgangsfigur zur Normalform und dann rückwärts von der Normalform zur Zielfigur.

**Die Normalform:** Wir wählen für Figuren ein Maß, nach dem man sie ordnen kann: die **sortierte Liste ihrer Plättchen**
(sortiert nach `x`, bei Gleichstand nach `y`). Eine Figur ist „kleiner“, wenn ihre Liste kleiner ist, die Plättchen also weiter links
oben liegen. Das „Aufklappen“ ist ein **Abstieg**: Wir führen immer die Drehung aus, die zur kleinsten Figur führt, solange diese
kleiner ist als die aktuelle. Wenn keine Drehung mehr kleiner macht, sind wir bei der Normalform. Das Verfahren endet immer, denn es
gibt nur endlich viele Figuren.

**Es kann schiefgehen:** Die Normalform ist nicht immer eindeutig. Zwei überführbare Figuren können in verschiedenen Sackgassen
enden. Darum probieren wir **acht Ordnungen**. Dazu spiegeln und vertauschen wir vor dem Vergleichen die Koordinaten aller Plättchen
auf eine der acht möglichen Arten (`(x, y)`, `(−x, y)`, `(x, −y)`, `(−x, −y)` und das alles mit vertauschten `x` und `y`). Mit jeder
Ordnung „klappt“ man anders auf. Stimmen für **eine** Ordnung die Normalformen beider Figuren überein, haben wir einen Weg.
Stimmen sie für keine überein, wissen wir nichts: Das Verfahren kann **nur Ja** beweisen.

In [ ]:
def ordnungsschluessel(figur, ordnung):
    liste = []
    for x, y in figur[0]:
        if ordnung >= 4:
            x, y = y, x                                    # x und y vertauschen
        if ordnung % 2 == 1:
            x = -x
        if (ordnung // 2) % 2 == 1:
            y = -y
        liste.append((x, y))
    liste.sort()
    return liste


def klappe_auf(figur, ordnung):
    """Der Weg (Liste von Figuren) von figur zu einer Normalform: immer die Drehung, die die Figur nach der Ordnung am kleinsten macht."""
    weg = [figur]
    while True:
        bester_schluessel = ordnungsschluessel(figur, ordnung)
        beste_figur = None
        for neue_figur, markiert in alle_zuege(figur):
            schluessel = ordnungsschluessel(neue_figur, ordnung)
            if schluessel < bester_schluessel:
                bester_schluessel = schluessel
                beste_figur = neue_figur
        if beste_figur is None:
            return weg
        figur = beste_figur
        weg.append(figur)


def normalform_weg(figur_a, figur_b):
    """Ergebnis: Liste der Figuren von a nach b über eine gemeinsame Normalform, oder None."""
    for ordnung in range(8):
        weg_a = klappe_auf(figur_a, ordnung)
        weg_b = klappe_auf(figur_b, ordnung)
        if weg_a[-1] == weg_b[-1]:
            weg_b.reverse()
            return weg_a + weg_b[1:]
    return None


for name in ['02', '06', '07']:
    start, ziel = lies_figuren('beispieldaten/transform' + name + '.txt')
    weg = normalform_weg(start, ziel)
    if weg is None:
        print('transform' + name + ': keine gemeinsame Normalform gefunden')
    else:
        print('transform' + name + ':', len(start[0]), 'Plättchen, Weg mit', len(weg) - 1, 'Drehungen')

transform02: 11 Plättchen, Weg mit 4 Drehungen
transform06: 119 Plättchen, Weg mit 18 Drehungen
transform07: 200 Plättchen, Weg mit 46 Drehungen


Bei `transform06.txt` (eine spiralförmige Figur aus 119 Plättchen) und `transform07.txt` (ein Rückgrat aus 100 Plättchen mit je einem Blatt daneben)
funktioniert das Aufklappen, und die Breitensuche würde dort nie fertig. Der Weg ist **nicht unbedingt kurz**, er hat nur Drehungen, die
schon zur Normalform und zurück führen. Die Aufgabe fordert keinen kürzesten Weg.

Bei `transform07.txt` sieht man, warum der Abstieg richtig vorgeht: Das Umklappen **einer** Stelle klappt den ganzen Rest der Kette
mit um. Wer von der Zielfigur her sucht, müsste die Blätter in einer ganz bestimmten Reihenfolge umlegen. Der Abstieg zur
Normalform „sortiert“ sie von selbst, nämlich so, dass alle Blätter links sind.

## Schritt 7: Alles zusammen, Ausgabe und Nachprüfung

`entscheide(start, ziel)` kombiniert die Verfahren:

1. Sprechen die **Invarianten** dagegen: `n`.
2. Hat die Figur höchstens `KLEIN` Plättchen: **Breitensuche** (sie ist hier entweder ja oder nein, solange die Grenze nicht erreicht wird).
3. Hat die Figur mehr als `KLEIN` und höchstens `MITTEL` Plättchen (oder hat die Breitensuche aufgegeben): **Aufklappen** zur Normalform.
4. Sonst bleibt das Ergebnis `?` (**unbekannt**). Für `?` geben wir nichts weiter aus.

Die Schranken `KLEIN` (30) und `MITTEL` (200) schützen vor zu langen Rechnungen: Bei größeren Figuren dauert schon das Aufklappen
sehr lange. Bei `transform08.txt`, `transform12.txt` und `transform13.txt` fand es in unseren Versuchen nichts, bei `transform13.txt`
auch nicht nach mehr als sieben Minuten Rechenzeit. Dann kommen wir ohne Ergebnis zurück.

Die Ausgabe: Für einen Weg aus Figuren `f0, f1, …, fk` brauchen wir für jede Drehung `fi → f(i+1)` das **markierte Plättchen**. Das finden
wir, indem wir die Drehungen von `fi` erzeugen und die suchen, die zu `f(i+1)` führt (`finde_markierung`). Gedruckt wird die Figur
**vor** jeder Drehung mit dem `#` am markierten Plättchen.

Zur Kontrolle prüft `pruefe_weg`, unabhängig von der Suche, ob jeder Schritt eine **erlaubte Drehung** ist: Die nächste Figur steht in
der Liste der Drehungen der aktuellen Figur, und zwar mit dem angegebenen Plättchen. Am Ende muss die Zielfigur herauskommen.

In [ ]:
KLEIN = 30
MITTEL = 200


def finde_markierung(figur, naechste):
    for neue_figur, markiert in alle_zuege(figur):
        if neue_figur == naechste:
            return markiert
    return None


def entscheide(start, ziel):
    """Ergebnis: (urteil, weg, grund), urteil ist 'y', 'n' oder '?'. weg ist die Liste der Figuren oder None."""
    if start == ziel:
        return 'y', [start], 'gleich'
    if not sind_vielleicht_ueberfuehrbar(start, ziel):
        return 'n', None, 'Invarianten'
    if len(start[0]) <= KLEIN:
        urteil, weg = breitensuche(start, ziel, 200000)
        if urteil == 'ja':
            return 'y', weg, 'Breitensuche'
        if urteil == 'nein':
            return 'n', None, 'Breitensuche'
    if len(start[0]) <= MITTEL:
        weg = normalform_weg(start, ziel)
        if weg is not None:
            return 'y', weg, 'Normalform'
    return '?', None, 'offen'


def ausgabe_text(urteil, weg):
    if urteil != 'y':
        return urteil + '\n'
    zeilen = ['y', str(len(weg) - 1)]
    for i in range(len(weg) - 1):
        zeilen.append(zeichne(weg[i], finde_markierung(weg[i], weg[i + 1])))
        zeilen.append('')
    return '\n'.join(zeilen) + '\n'


def pruefe_weg(start, ziel, weg):
    if weg[0] != start or weg[-1] != ziel:
        return False
    for i in range(len(weg) - 1):
        if finde_markierung(weg[i], weg[i + 1]) is None:
            return False
    return True


start, ziel = lies_figuren('beispieldaten/transform02.txt')
urteil, weg, grund = entscheide(start, ziel)
print(ausgabe_text(urteil, weg))
print('Weg in Ordnung:', pruefe_weg(start, ziel, weg))

y
3
6 3
..o-o......
..|.|......
o.o.o-o-o-o
|...|......
#-o-o......

6 3
......o-#..
......|.|..
o-o-o-o.o.o
......|...|
......o-o-o

6 4
........o..
........|..
......o-o..
......|....
o-o-o-o...o
......|...|
......o-#-o


Weg in Ordnung: True


Das ist ein Weg mit drei Drehungen für `transform02.txt`, und er sieht aus wie die Beispielausgabe in den Unterlagen: Zuerst die
Ausgangsfigur mit dem markierten Plättchen, dann die Figur nach der ersten Drehung und so weiter. Der Weg ist nicht derselbe wie in
der Beispielausgabe, denn es gibt oft mehrere.

## Schritt 8: Alle Beispiele

Wir lassen das Verfahren auf alle Beispiele los und prüfen jeden gefundenen Weg. `?` bedeutet: Das Verfahren kann die Frage **nicht
entscheiden**. Das ist ehrlich gemeint, nicht „nein“.

In [ ]:
ergebnisse = {}
for name in ['0y', '0n'] + ['%02d' % i for i in range(1, 14)]:
    start, ziel = lies_figuren('beispieldaten/transform' + name + '.txt')
    urteil, weg, grund = entscheide(start, ziel)
    ergebnisse[name] = (start, ziel, urteil, weg, grund)
    text = 'transform' + name + ': ' + str(len(start[0])).rjust(4) + ' Plättchen, Ergebnis ' + urteil + ' (' + grund + ')'
    if urteil == 'y':
        text = text + ', ' + str(len(weg) - 1) + ' Drehungen, Weg in Ordnung: ' + str(pruefe_weg(start, ziel, weg))
    print(text)

transform0y:    5 Plättchen, Ergebnis y (Breitensuche), 1 Drehungen, Weg in Ordnung: True
transform0n:    5 Plättchen, Ergebnis n (Invarianten)
transform01:    5 Plättchen, Ergebnis n (Invarianten)
transform02:   11 Plättchen, Ergebnis y (Breitensuche), 3 Drehungen, Weg in Ordnung: True
transform03:   14 Plättchen, Ergebnis n (Breitensuche)
transform04: 2006 Plättchen, Ergebnis n (Invarianten)
transform05:   22 Plättchen, Ergebnis n (Breitensuche)
transform06:  119 Plättchen, Ergebnis y (Normalform), 18 Drehungen, Weg in Ordnung: True
transform07:  200 Plättchen, Ergebnis y (Normalform), 46 Drehungen, Weg in Ordnung: True
transform08:  201 Plättchen, Ergebnis ? (offen)
transform09: 8037 Plättchen, Ergebnis ? (offen)
transform10:    8 Plättchen, Ergebnis y (Breitensuche), 2 Drehungen, Weg in Ordnung: True
transform11:   16 Plättchen, Ergebnis y (Breitensuche), 4 Drehungen, Weg in Ordnung: True
transform12:  496 Plättchen, Ergebnis ? (offen)
transform13:  516 Plättchen, Ergebnis ? (offen

**Ergebnis und Vergleich:** Die offiziellen Lösungshinweise geben für jedes Beispiel an, ob es überführbar ist (für unsere Art der
Drehung, die Kantendrehung):

| Beispiel | Plättchen | Lösungshinweise | unser Ergebnis | wie |
|---|---|---|---|---|
| transform0y | 5 | ja | **y** | Breitensuche |
| transform0n | 5 | nein | **n** | Invarianten |
| transform01 | 5 | nein | **n** | Invarianten |
| transform02 | 11 | ja | **y** | Breitensuche |
| transform03 | 14 | nein | **n** | Breitensuche (alles durchsucht) |
| transform04 | 2006 | nein | **n** | Invarianten |
| transform05 | 22 | nein | **n** | Breitensuche (alles durchsucht) |
| transform06 | 119 | ja | **y** | Aufklappen |
| transform07 | 200 | ja | **y** | Aufklappen |
| transform08 | 201 | ja | ? | offen |
| transform09 | 8037 | ja | ? | offen |
| transform10 | 8 | ja | **y** | Breitensuche |
| transform11 | 16 | ja | **y** | Breitensuche |
| transform12 | 496 | ja | ? | offen |
| transform13 | 516 | nein | ? | offen |

Elf von fünfzehn Beispielen entscheidet das Verfahren, und alle elf Antworten stimmen mit den Lösungshinweisen überein. Bei den
übrigen vier (`transform08`, `09`, `12` und `13`) bleibt die Frage offen. Das sind die großen Figuren, bei denen weder die Suche noch
das einfache Aufklappen weiterkommt. Das ist ehrlicher als eine geratene Antwort.

## Schritt 9: Die vollständigen Ausgaben

Zuerst die vollständigen Ausgaben für die Beispiele, bei denen die Aufgabe sie in der Dokumentation verlangt (`transform0n`, `0y`, `01`,
`02`, `03`, `05` und `10`). Bei `n` folgt nichts weiter.

In [ ]:
for name in ['0n', '0y', '01', '02', '03', '05', '10']:
    start, ziel, urteil, weg, grund = ergebnisse[name]
    print('transform' + name + '_out.txt:')
    print(ausgabe_text(urteil, weg))

transform0n_out.txt:
n

transform0y_out.txt:
y
1
3 2
..#-o
..|..
o-o-o


transform01_out.txt:
n

transform02_out.txt:
y
3
6 3
..o-o......
..|.|......
o.o.o-o-o-o
|...|......
#-o-o......

6 3
......o-#..
......|.|..
o-o-o-o.o.o
......|...|
......o-o-o

6 4
........o..
........|..
......o-o..
......|....
o-o-o-o...o
......|...|
......o-#-o


transform03_out.txt:
n

transform05_out.txt:
n

transform10_out.txt:
y
2
5 2
o...#-o..
|...|.|..
o-o-o.o-o

5 3
o-o-#....
|...|....
o...o-o..
......|..
......o-o




Und die ersten beiden Zeilen der Ausgabe für **alle** Beispiele. Bei `?` ist das Ergebnis offen, die Aufgabe kennt dafür kein Format.

In [ ]:
for name in ['0y', '0n'] + ['%02d' % i for i in range(1, 14)]:
    start, ziel, urteil, weg, grund = ergebnisse[name]
    if urteil == '?':
        print('transform' + name + ': ? (offen)')
    else:
        zeilen = ausgabe_text(urteil, weg).split('\n')
        print('transform' + name + ':', ' '.join(zeilen[:2]).strip())

transform0y: y 1
transform0n: n
transform01: n
transform02: y 3
transform03: n
transform04: n
transform05: n
transform06: y 18
transform07: y 46
transform08: ? (offen)
transform09: ? (offen)
transform10: y 2
transform11: y 4
transform12: ? (offen)
transform13: ? (offen)


Die Ausgabe in eine Datei schreibt man mit:

```python
with open('transform02_out.txt', 'w', encoding='utf-8') as f:
    f.write(ausgabe_text(urteil, weg))
```

## Für Neugierige

**Wie ist die Aufgabe gemeint?** Die Aufgabe lässt zwei Auslegungen zu. Wir haben die **Kantendrehung** gewählt: Eine Teilfigur wird um
den Stift gedreht, an dem sie hängt. Bei der **Knotendrehung** greift man ein **Plättchen** und dreht alles, was daran hängt, um die
Gerade durch zwei gegenüberliegende Stifte des Plättchens. Die Lösungshinweise sagen für beide Auslegungen, welche Beispiele gehen. Sie
unterscheiden sich nur bei `transform13.txt`: Das geht mit der Knotendrehung, mit der Kantendrehung nicht. Unser Programm entscheidet
`transform13.txt` nicht, es beweist also weder das eine noch das andere.

**Warum ist das Problem schwer?** Der Zustandsraum ist riesig. Hat eine Figur zum Beispiel ein Rückgrat mit `m` Blättern, die man unabhängig
umlegen kann, gibt es `2^m` verschiedene Figuren. Die Lösungshinweise zeigen sogar, dass die Aufgabe im Allgemeinen **NP-schwer** ist.
Darum gibt es für beliebige Figuren kein schnelles Verfahren, und man ist auf Heuristiken angewiesen.

**Was die Lösungshinweise noch tun:**
- **Gerichtete Isomorphie:** Wenn zwei Figuren überführbar sind, muss es eine Zuordnung zwischen ihren Plättchen geben, die waagerechte
  und senkrechte Nachbarn beachtet. Die Farbverfeinerung liefert sie oft gleich mit, wenn jede Farbe nur einmal vorkommt. Damit weiß man,
  welches Plättchen wohin soll.
- **Gierige Suche:** Man zählt, wie viele Plättchen schon an der richtigen Stelle liegen, und führt die Drehung aus, die diese Zahl am
  meisten erhöht, mit Zurückgehen bei Sackgassen.
- **Aufklappen** und **Segmentierung** (unabhängige Teile getrennt behandeln), wie wir sie zum Teil benutzt haben.
- **Gauß-Jordan-Verfahren:** Hat man die Zuordnung, kann man ein Gleichungssystem über den Zahlen 0 und 1 aufstellen, das sagt, welche Teile
  gedreht werden müssen. Es sagt aber nicht, in welcher Reihenfolge, und ob sich dabei Plättchen überlappen. Es ist also nur ein Hinweis.

**Schneller werden:** Unsere Brückensuche ist einfach, aber langsam (für jeden Stift ein Flood Fill). Der Algorithmus von **Tarjan** findet alle
Brücken in einem einzigen Durchlauf. Eine **beidseitige Breitensuche** (von beiden Figuren aus, bis sich die Suchen treffen) braucht deutlich
weniger Zustände als die einseitige.

**MiniZinc:** Hier haben wir keinen Abschnitt, denn es geht um die Suche in einem riesigen Raum von Figuren, nicht um das Erfüllen von
Bedingungen auf einer festen Zahl von Variablen.